### Renewable Energy Share in Indonesia (2007-2025): descriptive analysis and short-term ARIMA forecast (2026–2030).

In [1]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

***Part 1: Settings.***

In [2]:
COUNTRY_NAME = "Indonesia"
START_YEAR = 2007
END_YEAR = 2025
FORECAST_HORIZON = 5
TEST_SIZE = 3

OUTPUT_DIR = Path("appendix_output")
OUTPUT_DIR.mkdir(exist_ok=True)

ACCESS_DATE = datetime.today().strftime("%B %d, %Y")

***Part 2: Dataset.***

In [3]:
RAW_DATA = [
    (2007, 3.96, "HEESI 2017, Table 1.7.2"),
    (2008, 4.37, "HEESI 2017, Table 1.7.2"),
    (2009, 4.35, "HEESI 2019, Table 1.6"),
    (2010, 5.42, "HEESI 2019, Table 1.6"),
    (2011, 3.77, "HEESI 2019, Table 1.6"),
    (2012, 3.92, "HEESI 2019, Table 1.6"),
    (2013, 4.96, "HEESI 2019, Table 1.6"),
    (2014, 5.32, "HEESI 2019, Table 1.6"),
    (2015, 4.97, "HEESI 2019, Table 1.6"),
    (2016, 6.13, "HEESI 2019, Table 1.6"),
    (2017, 6.66, "HEESI 2019, Table 1.6"),
    (2018, 8.61, "HEESI 2019, Table 1.6"),
    (2019, 9.18, "HEESI 2019, Table 1.6"),
    (2020, 10.94, "HEESI 2023, Table 1.6"),
    (2021, 11.83, "HEESI 2023, Table 1.6"),
    (2022, 12.30, "IESR"),
    (2023, 13.10, "IESR"),
    (2024, 14.65, "HEESI 2024, Table 1.6"),
    (2025, 15.75, "MEMR"),
]


def build_dataset():
    """Return the compiled renewables-share series as a dataframe."""
    df = pd.DataFrame(RAW_DATA, columns=["Year", "RE_share", "Source"])
    df["Year"] = df["Year"].astype(int)
    df["RE_share"] = df["RE_share"].astype(float)
    df = df.sort_values("Year").reset_index(drop=True)

    # Basic sanity checks (they replace the OWID load-and-filter steps).

    if df.empty:
        raise ValueError("Dataset is empty.")
    if df["Year"].duplicated().any():
        raise ValueError("Duplicate years detected in the compiled dataset.")
    if not df["Year"].between(START_YEAR, END_YEAR).all():
        raise ValueError(f"Dataset contains years outside [{START_YEAR}, {END_YEAR}].")
    return df

***Part 3: Descriptive Analysis.***

In [4]:
def descriptive_analysis(df):
    """Write summary statistics and a historical line chart to disk."""
    summary = df["RE_share"].describe().round(3)
    summary.to_csv(OUTPUT_DIR / "summary_statistics.csv", header=["value"])

    df_desc = df.copy()
    df_desc["annual_change_pp"] = df_desc["RE_share"].diff().round(3)
    df_desc.to_csv(OUTPUT_DIR / "historical_series_with_changes.csv", index=False)

    plt.figure(figsize=(10, 6))
    plt.plot(df["Year"], df["RE_share"], marker="o")
    plt.title(
        f"Renewable Energy Share in Total Primary Energy Supply in {COUNTRY_NAME} "
        f"({START_YEAR}-{END_YEAR})"
    )
    plt.xlabel("Year")
    plt.ylabel(
        "Percent of energy generated from renewables in total primary energy supply"
    )
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_3_1_historical_series.png", dpi=300)
    plt.close()

    return summary

***Part 4: Stationarity Test.***

In [5]:
def adf_test(series):
    """Return ADF test output as a plain dictionary."""
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "adf_statistic": float(result[0]),
        "p_value": float(result[1]),
        "used_lag": int(result[2]),
        "n_obs": int(result[3]),
        "critical_value_1pct": float(result[4]["1%"]),
        "critical_value_5pct": float(result[4]["5%"]),
        "critical_value_10pct": float(result[4]["10%"]),
    }

***Part 5: ARIMA Model Search.***

In [6]:
def search_arima_with_trend(
    train_series,
    p_values=(0, 1),
    d_values=(0,),
    q_values=(0, 1),
    trend="ct",
):
    """Grid-search ARIMA(p, 0, q) with constant + linear trend."""
    results = []
    for order in itertools.product(p_values, d_values, q_values):
        try:
            fitted = ARIMA(train_series, order=order, trend=trend).fit()
            results.append(
                {
                    "p": order[0],
                    "d": order[1],
                    "q": order[2],
                    "trend": trend,
                    "aic": fitted.aic,
                    "bic": fitted.bic,
                }
            )
        except Exception as exc:
            print(f"  Skipped ARIMA{order} trend={trend}: {exc}")
    if not results:
        raise RuntimeError("No ARIMA specification converged on the training sample.")
    results_df = pd.DataFrame(results).sort_values("aic").reset_index(drop=True)
    results_df.to_csv(OUTPUT_DIR / "arima_aic_table.csv", index=False)

    best_order = (
        int(results_df.loc[0, "p"]),
        int(results_df.loc[0, "d"]),
        int(results_df.loc[0, "q"]),
    )
    best_trend = results_df.loc[0, "trend"]
    return (best_order, best_trend), results_df

***Part 6: Out-of-sample Validation.***

In [7]:
def validate_model(series, order_trend, test_size=TEST_SIZE):
    """Hold out the last `test_size` years and compute error metrics."""
    order, trend = order_trend

    if len(series) <= test_size + 3:
        raise ValueError("Series too short for the chosen test size.")
    train = series.iloc[:-test_size]
    test = series.iloc[-test_size:]

    fitted = ARIMA(train, order=order, trend=trend).fit()
    forecast = fitted.forecast(steps=test_size)

    # Use raw arrays so pandas index alignment cannot turn things into NaN.

    actual = np.asarray(test.values, dtype=float)
    predicted = np.asarray(forecast, dtype=float)

    mae = float(mean_absolute_error(actual, predicted))
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    with np.errstate(divide="ignore", invalid="ignore"):
        safe_actual = np.where(actual == 0, np.nan, actual)
        mape = float(np.nanmean(np.abs((actual - predicted) / safe_actual)) * 100)
    validation_df = pd.DataFrame(
        {
            "Year": test.index,
            "Actual": actual,
            "Forecast": predicted,
            "Absolute_Error": np.abs(actual - predicted),
        }
    )
    validation_df.to_csv(OUTPUT_DIR / "validation_results.csv", index=False)

    # Small validation chart (useful as Figure 3.2 in the thesis).

    plt.figure(figsize=(8, 5))
    plt.plot(train.index, train.values, marker="o", label="Training data")
    plt.plot(test.index, actual, marker="o", label="Actual (holdout)")
    plt.plot(
        test.index, predicted, marker="o", linestyle="--", label="Forecast on holdout"
    )
    plt.title("Validation on 2023-2025 holdout")
    plt.xlabel("Year")
    plt.ylabel(
        "PPercent of energy generated from renewables in total primary energy supply"
    )
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_3_2_validation.png", dpi=300)
    plt.close()

    metrics = {
        "order": order,
        "trend": trend,
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
    }
    pd.DataFrame([metrics]).to_csv(
        OUTPUT_DIR / "forecast_accuracy_metrics.csv", index=False
    )
    return metrics, validation_df

***Part 7: Final Forecast.***

In [8]:
def final_forecast(series, order_trend, horizon=FORECAST_HORIZON):
    """Refit on the full sample and forecast `horizon` years ahead."""
    order, trend = order_trend
    fitted = ARIMA(series, order=order, trend=trend).fit()

    forecast_res = fitted.get_forecast(steps=horizon)
    forecast_mean = np.asarray(forecast_res.predicted_mean, dtype=float)
    conf_int = forecast_res.conf_int()
    lower_ci = np.asarray(conf_int.iloc[:, 0].values, dtype=float)
    upper_ci = np.asarray(conf_int.iloc[:, 1].values, dtype=float)

    future_years = np.arange(series.index.max() + 1, series.index.max() + horizon + 1)

    forecast_df = pd.DataFrame(
        {
            "Year": future_years,
            "Forecast_RE_share": forecast_mean,
            "Lower_95_CI": lower_ci,
            "Upper_95_CI": upper_ci,
        }
    )
    forecast_df.to_csv(OUTPUT_DIR / "table_3_4_forecast_values.csv", index=False)

    with open(OUTPUT_DIR / "arima_model_summary.txt", "w", encoding="utf-8") as f:
        f.write(str(fitted.summary()))
    return fitted, forecast_df

***Part 8: Forecast Chart.***

In [9]:
def plot_forecast(df_hist, forecast_df):
    """Plot historical observations together with the short-term forecast."""
    plt.figure(figsize=(10, 6))

    plt.plot(df_hist["Year"], df_hist["RE_share"], marker="o", label="Historical data")

    plt.plot(
        forecast_df["Year"],
        forecast_df["Forecast_RE_share"],
        marker="o",
        linestyle="--",
        label="Forecast",
    )

    plt.fill_between(
        forecast_df["Year"],
        forecast_df["Lower_95_CI"],
        forecast_df["Upper_95_CI"],
        alpha=0.2,
        label="95% confidence interval",
    )

    plt.title(
        f"Renewable Energy Share in Total Primary Energy Supply in {COUNTRY_NAME}: "
        f"Historical Trend and Forecast"
    )
    plt.xlabel("Year")
    plt.ylabel(
        "Percent of energy generated from renewables in total primary energy supply"
    )
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_3_3_forecast.png", dpi=300)
    plt.close()

***Part 9: Main Workflow.***

In [10]:
def main():
    print("Building compiled dataset...")
    df = build_dataset()
    df.to_csv(OUTPUT_DIR / "clean_dataset_2007_2025.csv", index=False)
    df[["Year", "RE_share"]].to_csv(
        OUTPUT_DIR / "table_3_3_historical_values.csv", index=False
    )

    print("\nCompiled series:")
    print(df.to_string(index=False))

    print("\nDescriptive statistics:")
    summary = descriptive_analysis(df)
    print(summary)

    series = pd.Series(df["RE_share"].values, index=df["Year"].values)

    print("\nADF stationarity test...")
    adf_levels = adf_test(series)
    adf_diff = adf_test(series.diff())
    with open(OUTPUT_DIR / "adf_results.txt", "w", encoding="utf-8") as f:
        f.write("ADF on levels:\n")
        for k, v in adf_levels.items():
            f.write(f"  {k}: {v}\n")
        f.write("\nADF on first difference:\n")
        for k, v in adf_diff.items():
            f.write(f"  {k}: {v}\n")
    print(f"  Levels p-value:     {adf_levels['p_value']:.4f}")
    print(f"  First diff p-value: {adf_diff['p_value']:.4f}")

    print("\nSearching ARIMA orders with deterministic trend...")
    train_series = series.iloc[:-TEST_SIZE] if len(series) > TEST_SIZE else series
    best, comparison_df = search_arima_with_trend(train_series)
    print(f"  Best specification: order={best[0]} trend={best[1]}")
    print(comparison_df.head(5).to_string(index=False))

    print("\nValidating on holdout sample...")
    metrics, validation_df = validate_model(series, best, test_size=TEST_SIZE)
    print(f"  MAE  = {metrics['mae']:.3f}")
    print(f"  RMSE = {metrics['rmse']:.3f}")
    print(f"  MAPE = {metrics['mape']:.2f}%")

    print("\nRefitting on the full sample and forecasting 2026-2030...")
    fitted_model, forecast_df = final_forecast(series, best, horizon=FORECAST_HORIZON)
    print(forecast_df.round(3).to_string(index=False))

    print("\nPlotting forecast chart...")
    plot_forecast(df, forecast_df)

    print(f"\nDone. All files saved in: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

Building compiled dataset...

Compiled series:
 Year  RE_share                  Source
 2007      3.96 HEESI 2017, Table 1.7.2
 2008      4.37 HEESI 2017, Table 1.7.2
 2009      4.35   HEESI 2019, Table 1.6
 2010      5.42   HEESI 2019, Table 1.6
 2011      3.77   HEESI 2019, Table 1.6
 2012      3.92   HEESI 2019, Table 1.6
 2013      4.96   HEESI 2019, Table 1.6
 2014      5.32   HEESI 2019, Table 1.6
 2015      4.97   HEESI 2019, Table 1.6
 2016      6.13   HEESI 2019, Table 1.6
 2017      6.66   HEESI 2019, Table 1.6
 2018      8.61   HEESI 2019, Table 1.6
 2019      9.18   HEESI 2019, Table 1.6
 2020     10.94   HEESI 2023, Table 1.6
 2021     11.83   HEESI 2023, Table 1.6
 2022     12.30                    IESR
 2023     13.10                    IESR
 2024     14.65   HEESI 2024, Table 1.6
 2025     15.75                    MEMR

Descriptive statistics:
count    19.000
mean      7.905
std       4.004
min       3.770
25%       4.665
50%       6.130
75%      11.385
max      15.750


C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, for

  Best specification: order=(1, 0, 0) trend=ct
 p  d  q trend       aic       bic
 1  0  0    ct 46.778544 49.868899
 1  0  1    ct 48.768007 52.630950
 0  0  1    ct 52.576459 55.666814
 0  0  0    ct 58.155873 60.473639

Validating on holdout sample...


C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, for

  MAE  = 1.462
  RMSE = 1.647
  MAPE = 9.74%

Refitting on the full sample and forecasting 2026-2030...
 Year  Forecast_RE_share  Lower_95_CI  Upper_95_CI
 2026             16.232       14.713       17.751
 2027             16.740       14.745       18.735
 2028             17.271       14.992       19.550
 2029             17.820       15.355       20.285
 2030             18.386       15.794       20.977

Plotting forecast chart...


C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\Лиза\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, for


Done. All files saved in: C:\Users\Лиза\AppData\Local\Programs\Microsoft VS Code\appendix_output
